In [1]:
rm(list = ls())

# check and install CRAN packages
cran_packages <- c("tidyverse", "plyr", "openxlsx", "scales", "ggplot2")

for (pkg in cran_packages) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg, dependencies = TRUE)
  }
}

lapply(cran_packages, library, character.only = TRUE)


── Attaching core tidyverse packages ───────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.2
✔ purrr     1.2.0     
── Conflicts ─────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
------------------------------------------------------------------------------

You have loaded plyr after dplyr - this is likely to cause problems.
If you need functions from both plyr and dplyr, please load plyr first, then dplyr:
library(plyr); library(dplyr)

------------------------------------------------------------------------------


Attaching package: ‘plyr’

[[1]]
 [1] "lubridate" "forcats"   "stringr"   "dplyr"     "purrr"     "readr"    
 [7] "tidyr"     "tibble"    "ggplot2"   "tidyverse" "repr"      "stats"    
[13] "graphics"  "grDevices" "utils"     "datasets"  "methods"   "base"     

[[2]]
 [1] "plyr"      "lubridate" "forcats"   "stringr"   "dplyr"     "purrr"    
 [7] "readr"     "tidyr"     "tibble"    "ggplot2"   "tidyverse" "repr"     
[13] "stats"     "graphics"  "grDevices" "utils"     "datasets"  "methods"  
[19] "base"     

[[3]]
 [1] "openxlsx"  "plyr"      "lubridate" "forcats"   "stringr"   "dplyr"    
 [7] "purrr"     "readr"     "tidyr"     "tibble"    "ggplot2"   "tidyverse"
[13] "repr"      "stats"     "graphics"  "grDevices" "utils"     "datasets" 
[19] "methods"   "base"     

[[4]]
 [1] "scales"    "openxlsx"  "plyr"      "lubridate" "forcats"   "stringr"  
 [7] "dplyr"     "purrr"     "readr"     "tidyr"     "tibble"    "ggplot2"  
[13] "tidyverse" "repr"      "stats"     "graphics"  "grDevices" "utils"    
[19] "datasets"  "methods"   "base"     

[[5]]
 [1] "scales"    "openxlsx"  "plyr"      "lubridate" "forcats"   "stringr"  
 [7] "dplyr"     "purrr"     "readr"     "tidyr"     "tibble"    "ggplot2"  
[13] "tidyverse" "repr"      "stats"     "graphics"  "grDevices" "utils"    
[19] "datasets"  "methods"   "base"

# Figure S1a : Size distribution of sRNAs

In [2]:
# load sRNA data
data <- read.delim( "./lib/summary_clean_reads.txt", sep = " ", header = FALSE, stringsAsFactors = FALSE)
colnames(data)[c(1,3,4)] <- c("acc", "size_raw", "ratio_raw")
data[1:4,]

,acc,V2,size_raw,ratio_raw,V5
,<chr>,<chr>,<chr>,<chr>,<int>
1,An1,clean,18;19;20;21;22;23;24;25;26;27;28,102399;153370;211338;469367;274427;329665;782106;92873;72624;62894;63130,2614193
2,An2,clean,18;19;20;21;22;23;24;25;26;27;28,79013;111632;148015;322617;185636;221482;510588;73246;61368;54948;55302,1823847
3,Col1,clean,18;19;20;21;22;23;24;25;26;27;28,216987;297447;346337;714091;628295;383727;854424;423866;328277;251737;241817,4687005
4,Col2,clean,18;19;20;21;22;23;24;25;26;27;28,250146;379276;435199;1270903;1152268;577133;1631716;731494;407311;273821;265726,7374993


In [3]:
data <- data %>%
  mutate(
    library = case_when(
      str_detect(acc, "An")  ~ "An-1",
      str_detect(acc, "Col") ~ "Col-0",
      str_detect(acc, "Ct")  ~ "Ct-1",
      str_detect(acc, "Cvi") ~ "Cvi-1",
      str_detect(acc, "Eri") ~ "Eri-1",
      str_detect(acc, "Kyo") ~ "Kyo-1",
      str_detect(acc, "Ler") ~ "Ler-0",
      str_detect(acc, "Sha") ~ "Sha",
      TRUE ~ NA_character_
    )
  )


In [4]:
temp <- do.call(
  rbind,
  lapply(seq_len(nrow(data)), function(i) {
    size  <- as.numeric(strsplit(data[i,3], ";")[[1]])
    count <- as.numeric(strsplit(data[i,4], ";")[[1]])
    ratio <- count / sum(count)
    
    data.frame(
      acc = data[i,1],
      library = data$library[i],
      size = size,
      ratio = ratio
    )
  })
)

temp[1:4,]

,acc,library,size,ratio
,<chr>,<chr>,<dbl>,<dbl>
1,An1,An-1,18,0.03917041
2,An1,An-1,19,0.05866820
3,An1,An-1,20,0.08084254
4,An1,An-1,21,0.17954566


In [5]:
data_summary <- function(data, varname, groupnames) {
  summary_func <- function(x, col) {
    data.frame(
      Percentage = mean(x[[col]], na.rm = TRUE),
      sd = sd(x[[col]], na.rm = TRUE)
    )
  }
  
  ddply(data, groupnames, summary_func, varname)
}

df2 <- data_summary(
  data = temp,
  varname = "ratio",
  groupnames = c("library", "size")
)

colnames(df2) <- c("Accession", "Size", "Percentage", "sd")

df2$Size <- factor(df2$Size)
df2$Percentage <- df2$Percentage * 100

In [6]:

p <- ggplot(df2, aes(x = Size, y = Percentage, fill = Accession)) +
  geom_bar(
    stat = "identity",
    position = position_dodge(0.9),
    width = 0.9,
    linewidth = 0.2
  ) +
  scale_fill_manual(
    values = alpha(c(
      "An-1" = "#0000FF",
      "Col-0" = "#FFA500",
      "Ct-1" = "#FF0000",
      "Cvi-1" = "#800000",
      "Eri-1" = "#000080",
      "Kyo-1" = "#008000",
      "Ler-0" = "#800080",
      "Sha"   = "#696969"
    ), 0.8)
  ) +
  scale_x_discrete(breaks = as.character(18:28)) +
  ylim(0, 40) +
  labs(
    x = "sRNA length (nucleotide, nt)",
    y = "Percentage (%)"
  ) +
  theme_classic() +
  theme(
    axis.text = element_text(color = "black"),
    axis.ticks = element_line(color = "black"),
    legend.key.size = unit(0.5, "cm"),
    legend.text = element_text(size = 10)
  )

ggsave(p, file = "size_distribution_of_sRNAs.pdf", width = 4.6, height = 2.6)


# Figure S1b: Size distribution of PPR-siRNAs

In [7]:
data <- read.delim( "./lib/summary_ppr-siRNA_count.txt", sep = " ", header = FALSE, stringsAsFactors = FALSE)
colnames(data)[c(1,3,4)] <- c("acc", "size_raw", "ratio_raw")
data[1:4,]

,acc,V2,size_raw,ratio_raw,V5
,<chr>,<chr>,<chr>,<chr>,<int>
1,An1,count,18;19;20;21;22;23;24;25;26;27;28,326;522;868;6895;1342;212;350;112;96;104;88,10915
2,An2,count,18;19;20;21;22;23;24;25;26;27;28,204;361;570;4585;905;158;227;86;82;97;78,7353
3,Col2w1,count,18;19;20;21;22;23;24;25;26;27;28,334;648;717;10342;1872;243;410;206;203;200;176,15351
4,Col2w2,count,18;19;20;21;22;23;24;25;26;27;28,526;1075;1318;21115;3779;389;761;223;201;220;252,29859


In [8]:
data <- data %>%
  mutate(
    library = case_when(
      str_detect(acc, "An")  ~ "An-1",
      str_detect(acc, "Col") ~ "Col-0",
      str_detect(acc, "Ct")  ~ "Ct-1",
      str_detect(acc, "Cvi") ~ "Cvi-1",
      str_detect(acc, "Eri") ~ "Eri-1",
      str_detect(acc, "Kyo") ~ "Kyo-1",
      str_detect(acc, "Ler") ~ "Ler-0",
      str_detect(acc, "Sha") ~ "Sha",
      TRUE ~ NA_character_
    )
  )


In [9]:
temp <- do.call(
  rbind,
  lapply(seq_len(nrow(data)), function(i) {
    size  <- as.numeric(strsplit(data[i,3], ";")[[1]])
    count <- as.numeric(strsplit(data[i,4], ";")[[1]])
    ratio <- count / sum(count)
    
    data.frame(
      acc = data[i,1],
      library = data$library[i],
      size = size,
      ratio = ratio
    )
  })
)



In [10]:
data_summary <- function(data, varname, groupnames) {
  summary_func <- function(x, col) {
    data.frame(
      Percentage = mean(x[[col]], na.rm = TRUE),
      sd = sd(x[[col]], na.rm = TRUE)
    )
  }
  
  ddply(data, groupnames, summary_func, varname)
}

df2 <- data_summary(
  data = temp,
  varname = "ratio",
  groupnames = c("library", "size")
)

colnames(df2) <- c("Accession", "Size", "Percentage", "sd")

df2$Size <- factor(df2$Size)
df2$Percentage <- df2$Percentage * 100

In [11]:

p <- ggplot(df2, aes(x = Size, y = Percentage, fill = Accession)) +
  geom_bar(
    stat = "identity",
    position = position_dodge(0.9),
    width = 0.9,
    linewidth = 0.2
  ) +
  scale_fill_manual(
    values = alpha(c(
      "An-1" = "#0000FF",
      "Col-0" = "#FFA500",
      "Ct-1" = "#FF0000",
      "Cvi-1" = "#800000",
      "Eri-1" = "#000080",
      "Kyo-1" = "#008000",
      "Ler-0" = "#800080",
      "Sha"   = "#696969"
    ), 0.8)
  ) +
  scale_x_discrete(breaks = as.character(18:28)) +
  ylim(0, 80) +
  labs(
    x = "sRNA length (nucleotide, nt)",
    y = "Percentage (%)"
  ) +
  theme_classic() +
  theme(
    axis.text = element_text(color = "black"),
    axis.ticks = element_line(color = "black"),
    legend.key.size = unit(0.5, "cm"),
    legend.text = element_text(size = 10)
  )

ggsave(p, file = "size_distribution_of_ppr-siRNAs.pdf", width = 4.6, height = 2.6)


# Figure S1c

In [12]:
data <- read.delim("./lib/ppr_homolog_variations.txt", sep = "\t", header = TRUE, check.names = FALSE) 

data$Class <- factor(data$Class, levels=c("siRNA-PPR", "non-siRNA-PPR"))
data$Accession <- factor(data$Accession, levels=c("An-1", "Col-0", "Ct-1", "Cvi-1", "Eri-1", "Kyo-1", "Ler-0", "Sha"))
data$Homolog <- factor(data$Homolog, levels=c("1", ">1"))
data$Ratio <- as.numeric(data$Ratio)

sum_table <- aggregate(Number ~ Accession + Class, data = data, sum)
sum_table$x_label <- paste0(sum_table$Accession, " (n = ", sum_table$Number, ")")

# merge x_label back to original data
data <- merge(data, sum_table[, c("Accession", "Class", "x_label")],
              by = c("Accession", "Class"),
              all.x = TRUE)

In [13]:
p <- ggplot(data, aes(x = x_label, y = Ratio, fill = Homolog)) +
  geom_bar(
    stat = "identity",
    width = 0.6,
    color = "black"
  ) +
  facet_grid(
    ~ Class,
    scales = "free_x",
    space = "free_x"
  ) +
  scale_fill_manual(
    values = c(">1" = "#EE6666", "1" = "grey")
  ) +
  scale_y_continuous(
    breaks = c(0, 0.25, 0.5, 0.75, 1),
    labels = c("0", "25", "50", "75", "100")
  ) +
  labs(x = NULL, y = "Percentage (%)") +
  theme_bw() +
  theme(
    panel.grid = element_blank(),
    panel.border = element_blank(),
    panel.background = element_blank(),
    axis.line = element_line(linewidth = 0.4, colour = "black"),
    axis.text = element_text(colour = "black", size = 10),
    axis.text.x = element_text(angle = 45, vjust = 1, hjust = 1),
    axis.ticks = element_line(colour = "black"),
    strip.background = element_blank(),
    strip.text = element_blank(),
    legend.position = "bottom"
  )

ggsave(p, file = "percentage_homolog_ppr.pdf", width = 5, height = 3.5)

In [14]:
sessionInfo()

R version 4.5.1 (2025-06-13)
Platform: aarch64-apple-darwin23.6.0
Running under: macOS Sonoma 14.5

Matrix products: default
BLAS:   /opt/homebrew/Cellar/openblas/0.3.30/lib/libopenblasp-r0.3.30.dylib 
LAPACK: /opt/homebrew/Cellar/r/4.5.1/lib/R/lib/libRlapack.dylib;  LAPACK version 3.12.1

locale:
[1] en_GB.UTF-8/en_GB.UTF-8/en_GB.UTF-8/C/en_GB.UTF-8/en_GB.UTF-8

time zone: America/Chicago
tzcode source: internal

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] scales_1.4.0     openxlsx_4.2.8.1 plyr_1.8.9       lubridate_1.9.4 
 [5] forcats_1.0.1    stringr_1.6.0    dplyr_1.1.4      purrr_1.2.0     
 [9] readr_2.1.6      tidyr_1.3.2      tibble_3.3.0     ggplot2_4.0.1   
[13] tidyverse_2.0.0  repr_1.1.7      

loaded via a namespace (and not attached):
 [1] gtable_0.3.6       jsonlite_2.0.0     compiler_4.5.1     crayon_1.5.3      
 [5] tidyselect_1.2.1   Rcpp_1.1.0         zip_2.3.3          IRdisplay_1.1